# 02: Train XGBoost Trip Time Model

Load training_data.csv from Notebook 01 (now with direction balanced, Sunday+Monday data), train XGBoost with `direction` as an additional feature, and export model + encoders to Google Drive.

In [ ]:
!pip install pandas numpy scikit-learn xgboost matplotlib joblib

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
import matplotlib.pyplot as plt
import joblib
from google.colab import files

print("Upload training_data.csv from Notebook 01:")
uploaded = files.upload()
df = pd.read_csv(list(uploaded.keys())[0])
print(f"Loaded {len(df)} records")
print(f"Direction balance: {df['direction'].value_counts().to_dict()}")
df.head()

In [ ]:
# Encode categorical features
cat_cols = ["route", "zone_type"]
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col])
    encoders[col] = le

# direction is already numeric (0/1), keep as-is
# day_of_week is already numeric (0-6), keep as-is

feature_cols = [
    "distance_km", "hour", "day_of_week", "is_holiday",
    "direction", "route_enc", "zone_type_enc"
]
X = df[feature_cols].copy()
y = df["trip_time_s"]

print("Features:", feature_cols)
print(f"X shape: {X.shape}, y mean: {y.mean():.1f}s")

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train)}, Val: {len(X_val)}")

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=1,
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=20)

In [ ]:
y_pred = model.predict(X_val)
mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(f"MAE: {mae:.1f}s")
print(f"RMSE: {rmse:.1f}s")
print(f"Baseline (mean predictor): {y_val.mean():.1f}s")

# Feature importance
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)
print("\n=== Feature importance ===")
print(importance.to_string(index=False))

# Residual plot
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.scatter(y_val, y_pred, alpha=0.3, s=10)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], "r--")
plt.xlabel("Actual (s)")
plt.ylabel("Predicted (s)")
plt.title("Prediction vs Actual")

plt.subplot(1, 2, 2)
residuals = y_val - y_pred
plt.hist(residuals, bins=50)
plt.xlabel("Residual (s)")
plt.title(f"Residuals (MAE={mae:.0f}s, RMSE={rmse:.0f}s)")
plt.tight_layout()
plt.show()

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/parkgo_gtfs_model"
import os
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(save_dir, "bus_trip_time_model.pkl")
joblib.dump(model, model_path)
print(f"Model saved to {model_path}")

for name, le in encoders.items():
    enc_path = os.path.join(save_dir, f"encoder_{name}.pkl")
    joblib.dump(le, enc_path)
    print(f"Encoder {name} saved")

feat_path = os.path.join(save_dir, "feature_cols.pkl")
joblib.dump(feature_cols, feat_path)
print("Feature columns saved")

# Also export feature list as txt for reference
with open(os.path.join(save_dir, "feature_cols.txt"), "w") as f:
    f.write("\n".join(feature_cols))
print("Feature columns TXT saved")

In [ ]:
# Quick demo: predict a single segment
def predict_trip_time(route_name, distance_km, hour, dow, is_holiday, direction, zone):
    route_enc = encoders["route"].transform([route_name])[0]
    zone_enc = encoders["zone_type"].transform([zone])[0]
    features = pd.DataFrame([[
        distance_km, hour, dow, is_holiday, direction, route_enc, zone_enc
    ]], columns=feature_cols)
    pred = model.predict(features)[0]
    return round(pred, 1)

# MT4, 0.5km segment
print("Weekday 8am dir 0:          {:.1f}s".format(predict_trip_time('MT4', 0.5, 8, 0, 0, 0, 'old_town')))
print("Weekday 8am dir 0 (bridge): {:.1f}s".format(predict_trip_time('MT4', 0.5, 8, 0, 0, 0, 'bridge')))
print("Weekday 8am dir 1:          {:.1f}s".format(predict_trip_time('MT4', 0.5, 8, 0, 0, 1, 'old_town')))
print("Weekday midnight:           {:.1f}s".format(predict_trip_time('MT4', 0.5, 2, 0, 0, 0, 'old_town')))
print("Weekend 8am:                {:.1f}s".format(predict_trip_time('MT4', 0.5, 8, 6, 0, 0, 'old_town')))